# A7: Full-Text Search Extension

Implement your alternative pipeline in this file. Much of the code should be similar to your solution to Part 1). Use
ElasticSearch to do stopword filtering and lemmatization here.

In [5]:
from xml.etree import ElementTree as ET
from elasticsearch import Elasticsearch, helpers

## Step 1: Parse the XML file (no nltk preprocessing this time)

In [6]:
def parse_pubmed_xml_raw(file_path):
    """
    Parse a PubMed XML file into a list of dicts with keys:
    'pmid', 'title', 'abstract'.
    Raw (unprocessed) text is stored — ElasticSearch will handle
    stopword removal and lemmatization via its analysis pipeline.
    """
    tree = ET.parse(file_path)
    root = tree.getroot()
    
    all_docs = []
    
    for article in root.findall('.//PubmedArticle'):
        pmid_elem = article.find('.//PMID')
        pmid = pmid_elem.text.strip() if pmid_elem is not None else None
        
        title_elem = article.find('.//ArticleTitle')
        title = ''.join(title_elem.itertext()).strip() if title_elem is not None else ''
        
        abstract_elems = article.findall('.//AbstractText')
        abstract_raw = ' '.join(
            ''.join(elem.itertext()).strip()
            for elem in abstract_elems
            if elem is not None
        ).strip()
        
        if pmid is not None:
            all_docs.append({
                'pmid': pmid,
                'title': title,
                'abstract': abstract_raw
            })
    
    return all_docs


XML_FILE = 'pubmed25n0001 (2).xml'
all_docs = parse_pubmed_xml_raw(XML_FILE)
print(f'Parsed {len(all_docs)} documents.')

Parsed 30000 documents.


## Step 2: Create an ElasticSearch index with a custom analyzer

We configure ElasticSearch to apply:
- **stop** token filter — removes English stopwords
- **snowball** (or `english` stemmer) token filter — handles lemmatization/stemming

See: https://www.elastic.co/docs/reference/text-analysis/analysis-stop-tokenfilter

In [7]:
es = Elasticsearch('http://localhost:9200')

EXT_INDEX = 'pubmed_ext'

# Custom index settings: define an analyzer that does stopword removal + stemming
index_settings = {
    'settings': {
        'analysis': {
            'filter': {
                'english_stop': {
                    'type': 'stop',
                    'stopwords': '_english_'   # built-in English stopword list
                },
                'english_stemmer': {
                    'type': 'stemmer',
                    'language': 'english'       # Porter stemmer (close to lemmatization)
                }
            },
            'analyzer': {
                'pubmed_analyzer': {
                    'type': 'custom',
                    'tokenizer': 'standard',
                    'filter': [
                        'lowercase',
                        'english_stop',
                        'english_stemmer'
                    ]
                }
            }
        }
    },
    'mappings': {
        'properties': {
            'pmid': {
                'type': 'keyword'
            },
            'title': {
                'type': 'text',
                'analyzer': 'pubmed_analyzer'
            },
            'abstract': {
                'type': 'text',
                'analyzer': 'pubmed_analyzer'
            }
        }
    }
}

# Delete existing index if present, then recreate
if es.indices.exists(index=EXT_INDEX):
    es.indices.delete(index=EXT_INDEX)
    print(f'Deleted existing index: {EXT_INDEX}')

es.indices.create(index=EXT_INDEX, body=index_settings)
print(f'Created index: {EXT_INDEX} with custom pubmed_analyzer')

# Bulk index all documents
actions = [
    {
        '_index': EXT_INDEX,
        '_id': doc['pmid'],
        '_source': doc
    }
    for doc in all_docs
]

success, failed = helpers.bulk(es, actions)
print(f'Indexed {success} documents successfully. Failed: {failed}')

es.indices.refresh(index=EXT_INDEX)

Deleted existing index: pubmed_ext
Created index: pubmed_ext with custom pubmed_analyzer
Indexed 30000 documents successfully. Failed: []


ObjectApiResponse({'_shards': {'total': 2, 'successful': 1, 'failed': 0}})

## Step 3: Verify — check how ElasticSearch analyzes the PMID=22 abstract

We use the `_analyze` API to inspect tokens that ElasticSearch stores for the document.

In [8]:
# Retrieve the raw document
result = es.get(index=EXT_INDEX, id='22')
doc_22 = result['_source']

print('PMID=22 — Raw Title:')
print(doc_22['title'])
print()
print('PMID=22 — Raw Abstract (stored as-is; analysis happens at search time):')
print(doc_22['abstract'])

# Show how ElasticSearch actually analyzes the abstract using our custom analyzer
if doc_22['abstract']:
    analyze_result = es.indices.analyze(
        index=EXT_INDEX,
        body={
            'analyzer': 'pubmed_analyzer',
            'text': doc_22['abstract']
        }
    )
    tokens = [t['token'] for t in analyze_result['tokens']]
    print()
    print('PMID=22 — Abstract after ElasticSearch stopword filtering + stemming:')
    print(' '.join(tokens))

if doc_22['title']:
    analyze_result_title = es.indices.analyze(
        index=EXT_INDEX,
        body={
            'analyzer': 'pubmed_analyzer',
            'text': doc_22['title']
        }
    )
    title_tokens = [t['token'] for t in analyze_result_title['tokens']]
    print()
    print('PMID=22 — Title after ElasticSearch stopword filtering + stemming:')
    print(' '.join(title_tokens))

PMID=22 — Raw Title:
[Demonstration of tumor inhibiting properties of a strongly immunostimulating low-molecular weight substance. Comparative studies with ifosfamide on the immuno-labile DS carcinosarcoma. Stimulation of the autoimmune activity for approx. 20 days by BA 1, a N-(2-cyanoethylene)-urea. Novel prophylactic possibilities].

PMID=22 — Raw Abstract (stored as-is; analysis happens at search time):
A report is given on the recent discovery of outstanding immunological properties in BA 1 [N-(2-cyanoethylene)-urea] having a (low) molecular mass M = 111.104. Experiments in 214 DS carcinosarcoma bearing Wistar rats have shown that BA 1, at a dosage of only about 12 percent LD50 (150 mg kg) and negligible lethality (1.7 percent), results in a recovery rate of 40 percent without hyperglycemia and, in one test, of 80 percent with hyperglycemia. Under otherwise unchanged conditions the reference substance ifosfamide (IF) -- a further development of cyclophosphamide -- applied without 